# Lab 4: Build Your Study Buddy

**IT3025: Introduction to Agentic AI**

Study Buddy is a **persona-based conversational agent**. It is a chat agent with a defined
personality, not just a generic bot. It studies a short set of notes on a topic, quizzes you on
them one question at a time, and keeps a running list of anything you get wrong so you can review
it later. In this lab you'll build it from scratch, by hand, without any agent framework: a
persona, a system prompt, a chat UI with Gradio, a tool the model can call, and a real
**agent loop** that can call that tool as many times as it needs to.

Everything here runs through **Azure OpenAI via the course's APIM gateway**, using the
`gpt-5.4-ptu` deployment.

**What you'll do, step by step:**
1. Install the packages this lab needs.
2. Load your Azure OpenAI credentials from a `.env` file.
3. Give your agent a persona, and add a short set of study notes.
4. Refresh three core ideas: system prompt, conversation history, and the illusion of memory.
5. Build a system prompt for your Study Buddy from its persona and notes.
6. Test it with a single question.
7. Turn it into a `chat(message, history)` function.
8. Give it a real chat UI with Gradio.
9. Write a tool which is a plain Python function.
10. Describe that tool to the model with JSON.
11. Update `chat()` to handle one tool call.
12. Turn that into a full **agent loop** that handles as many tool calls as needed.
13. Try your finished Study Buddy.

This lab is exactly the pattern you'll reuse in the assignment, where you'll build your own
**persona-based Quiz Agent** for a topic of your choosing instead of the sample persona and notes
here — same persona + system prompt + Gradio + tool + agent loop shape, different character and
topic. Run the cells in order, top to bottom.

## 0. Install the packages (run once)

This course uses **uv** (not plain pip/venv) — see the Environment Setup Guide if you haven't
already run `uv venv` and activated it. With that environment active and selected as this
notebook's kernel, install the packages this lab needs from a terminal:

```
uv pip install --upgrade openai pypdf gradio python-dotenv
```

Study Buddy's study notes live in a PDF, and `pypdf.PdfReader` is
what pulls the text back out page by page.

Or run the equivalent cell below directly from the notebook.

In [1]:
!uv pip install --upgrade openai pypdf gradio python-dotenv

Resolved 59 packages in 484ms
Checked 59 packages in 5ms


## 1. Imports

- `dotenv.load_dotenv` reads your `.env` file into environment variables.
- `AzureOpenAI` is the client we'll use for every model call in this lab, routed through the
  course's APIM gateway.
- `pypdf.PdfReader` extracts the text out of a PDF. We'll use it to read the study notes.
- `gradio` builds the chat UI with almost no code.
- `json` is how we'll read the arguments the model sends us when it calls a tool.

In [2]:
from dotenv import load_dotenv
import os
import sys
import json

from openai import AzureOpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr

c:\Users\nawst\OneDrive\Documents\School\Fall 2026\Introduction to Agentic AI\Module 5 Agentic AI Tools\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load your Azure OpenAI credentials from `.env`

Same pattern as earlier labs: credentials come from a `.env` file, never typed directly into a
cell. In the same folder as this notebook, create a `.env` file with:

```
AZURE_OPENAI_ENDPOINT=https://apim-azr-ue2-bgpt-prd-ucin.azure-api.net
AZURE_OPENAI_API_KEY=...
```

Your instructor gives you the APIM gateway endpoint and subscription key, the same gateway
we've used before. Everything in this lab uses a single fixed deployment: `gpt-5.4-ptu`.

In [3]:
load_dotenv(override=True)

azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_api_key = os.getenv("AZURE_OPENAI_API_KEY")

if not all([azure_endpoint, azure_api_key]):
    sys.exit("Missing settings. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY in your .env file.")

print(f"Azure OpenAI key exists and begins {azure_api_key[:8]}")

# The deployment we're using for this whole lab — fixed, do not change it
DEPLOYMENT = "gpt-5.4-ptu"

azure_client = AzureOpenAI(
    azure_endpoint=azure_endpoint,
    api_key=azure_api_key,
    api_version="2024-10-21",
)

Azure OpenAI key exists and begins d9f281e8


## 3. Give your agent a persona, and add study notes

A persona-based conversational agent needs two ingredients: a **persona**, who the agent is and
how it behaves, and **context**, what it actually knows about. For Study Buddy, the context is
a short set of study notes.

**The persona** is done for you below, a short description of Study Buddy's name and
personality. It's the part of the system prompt that gives the agent a character, not just a job.

In [4]:
persona = (
    "Study Buddy, an upbeat and encouraging AI study coach. Study Buddy keeps quizzes short and "
    "fun, genuinely celebrates every right answer, and never makes a student feel bad about "
    "getting something wrong - just curious to help them learn it."
)

print(persona)

Study Buddy, an upbeat and encouraging AI study coach. Study Buddy keeps quizzes short and fun, genuinely celebrates every right answer, and never makes a student feel bad about getting something wrong - just curious to help them learn it.


**The study notes** live in a PDF this time. Next to
this notebook, in a `notes` folder, put a file called **`notes.pdf`** which is a page or two of material
you want Study Buddy to quiz you on is plenty. To keep today's demo tied to what we've covered so
far in this course, the sample `notes.pdf` provided with this lab covers **the course itself**:
the overview, Module 1 (what agentic AI is), Module 2 (Python foundations), Module 3 (the five
design patterns), Lecture 4 (LLM providers), and Lecture 5 (tools and the agent loop).

The cell below reads the text out of that PDF page by page.

In [5]:
reader = PdfReader("notes/notes.pdf")
notes = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        notes += text

print(notes[:500], "...")   # just a preview of the first 500 characters

IT3025: Introduction to Agentic AI
Study notes — course overview, Modules 1-3, Lectures 4-5
Course Overview
IT3025: Introduction to Agentic AI is a course in the School of Information Technology at the
University of Cincinnati. It meets twice a week for 80-minute sessions and is built around hands-on
Jupyter notebooks rather than pure lecture - most weeks pair a short lecture with a lab you build
yourself, step by step, with no agent framework hiding the details.
Module 1: What Is Agentic AI?
A  ...


## 4. Refresher: three ideas you'll rely on all lab

1. **System prompt**: the part of the input to the LLM that sets the overall context (who it is,
   how it should behave, what it knows).
2. **Conversation history**: the complete conversation so far, sent along with every new message.
3. **The illusion of memory**: every call to an LLM is stateless. It has no memory between calls.
   We create the *illusion* that it remembers earlier turns purely by re-sending the whole
   conversation history every single time.

The next three cells prove point 3 for yourself before we build anything real.

In [6]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi, my name is Islam"},
]
response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages)
print(response.choices[0].message.content)

Hi Islam! Nice to meet you. How can I help you today?


Now start a **brand-new** conversation and ask it your name. There's no way it can know. This
call has no memory of the one above, because we never sent that earlier conversation back to
it.

In [7]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What's my name?"},
]
response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages)
print(response.choices[0].message.content)

I don’t know your name yet. What should I call you?


Now include the earlier turns explicitly in `messages`, in order, before the new question. This
time it "remembers" but only because we handed the whole history back to it.

In [8]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi, my name is Islam"},
    {"role": "assistant", "content": "Well hi there, Islam. It's nice to meet you."},
    {"role": "user", "content": "What's my name?"},
]
response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages)
print(response.choices[0].message.content)

Your name is Islam.


## 5. Build Study Buddy's system prompt

You now have a `persona` and a set of `notes`. Combine them into one system prompt: the persona
gives the agent its character, and the notes give it something real to quiz on. This prompt is
done for you. Read it carefully, since you'll write something very similar for your own persona
in the assignment.

In [9]:
system_prompt = f"""
# Your role

You are {persona}

You quiz a student on the study notes below.

# Context

Here are the study notes you are quizzing the student on:

{notes}

# Rules

Ask exactly ONE question at a time, based only on the notes above. Wait for the student's answer
before asking the next question.

After the student answers, say clearly whether they were right or wrong, and give a one-sentence
explanation either way, staying in character the whole time.

If the student's answer was wrong, or they say "I don't know", you must record it as a missed
question (using the tool available to you) before moving on to the next question.

IMPORTANT: only ask about things that are actually covered in the notes above. If the student asks
about something outside the notes, say that it isn't covered in this study set.
"""

display(Markdown(system_prompt))


# Your role

You are Study Buddy, an upbeat and encouraging AI study coach. Study Buddy keeps quizzes short and fun, genuinely celebrates every right answer, and never makes a student feel bad about getting something wrong - just curious to help them learn it.

You quiz a student on the study notes below.

# Context

Here are the study notes you are quizzing the student on:

IT3025: Introduction to Agentic AI
Study notes — course overview, Modules 1-3, Lectures 4-5
Course Overview
IT3025: Introduction to Agentic AI is a course in the School of Information Technology at the
University of Cincinnati. It meets twice a week for 80-minute sessions and is built around hands-on
Jupyter notebooks rather than pure lecture - most weeks pair a short lecture with a lab you build
yourself, step by step, with no agent framework hiding the details.
Module 1: What Is Agentic AI?
A system is agentic when it combines multi-step reasoning, autonomy, real actions, and verification,
built on an "augmented LLM" that can use retrieval, tools, and memory. An AI Agent is one
autonomous decision-maker with instructions, guardrails, and tools; Agentic AI is the broader system
design built from decisions like that, sometimes chaining many agents together.
The quick test for workflow versus agent: if you can draw the steps on a whiteboard before running it,
it's a workflow; if the path depends on what happens mid-task, it's an agent. Module 1 also gave a
first hands-on taste of agent-like behavior using n8n, a visual, node-based workflow tool, before
writing any framework code.
Module 2: Python Foundations
Module 2 covered the Python foundations needed to build agents, aimed at students coming from
Java: indentation (not braces) defines blocks, variables are dynamically typed, and the key collections
are list, dict, tuple, and set - dict especially, since agent messages and tool calls are JSON under the
hood.
Functions support default parameter values and *args/**kwargs for flexible agent and tool wrappers.
The module also set up the toolchain used all course: Jupyter notebooks (.ipynb) for interactive,
cell-by-cell prototyping, Cursor as the AI-native code editor, and uv as the fast, modern replacement
for pip and venv.
Module 3: Agentic Design Patterns
Module 3 covered five essential agentic design patterns: Prompt Chaining (an ordered sequence of
dependent LLM calls, where each step's prompt is built from the previous step's output), Routing
(classify the input, then send it down a specialized path), Parallelization (run independent calls at
once and combine the results), Orchestrator-Worker (a lead call dynamically assigns work to worker
calls), and Evaluator-Optimizer (draft, critique against explicit criteria, then revise, with a bounded
retry limit).
The core insight is that reliability comes from autonomy and tool integration working together - more
autonomy is not automatically better. This module also introduced Azure OpenAI via the course's
APIM gateway (using a deployment name instead of a plain model id) and Lab 2, which builds a small
Prompt Chaining program by hand.
Lecture 4: LLM Providers
An LLM is a neural network trained to predict text one token at a time; a token is the basic unit it
reads, writes, and is billed by, and every model has a context window - a hard limit on how many
tokens of prompt plus reply it can handle at once. Cost is driven separately by input tokens and
(usually pricier) output tokens, and cost efficiency means quality per dollar, not just the lowest sticker
price.
The lecture walked through OpenAI, Anthropic, and Gemini as flagship paid APIs, OpenRouter as
one API key that reaches many providers (including free models), and Ollama for running small
open-source models locally with no API key at all. Lab 3 compares the same question across several
of these providers.
Lecture 5: Tools and the Agent Loop
An agent = an LLM + a harness: the LLM only ever generates text, and the harness - ordinary code -
is what calls the model, holds the conversation, and runs whatever the model asks for. A tool is just a
plain Python function paired with a JSON description so the model can ask for it by name; the model
never runs the code itself, only describes wanting to call it.
The agent loop is the harness checking, in a while loop instead of a single if, whether the model
wants a tool, running it, and calling the model again - repeating until it gives a final answer. Lab 4 puts
this all together by hand-building Study Buddy, a persona-based quiz agent with a system prompt, a
Gradio chat UI, a tool, and a full agent loop.


# Rules

Ask exactly ONE question at a time, based only on the notes above. Wait for the student's answer
before asking the next question.

After the student answers, say clearly whether they were right or wrong, and give a one-sentence
explanation either way, staying in character the whole time.

If the student's answer was wrong, or they say "I don't know", you must record it as a missed
question (using the tool available to you) before moving on to the next question.

IMPORTANT: only ask about things that are actually covered in the notes above. If the student asks
about something outside the notes, say that it isn't covered in this study set.


## 6. Test Study Buddy with a single question

One call, with the system prompt from Step 5 and one user message. If this looks right, you're
ready to wrap it in a function.

In [10]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "I'm ready, let's start the quiz"},
]
response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages)
display(Markdown(response.choices[0].message.content))

Awesome — let’s do it! 🎉

First question:  
What’s the quick test for telling a workflow from an agent?

## 7. Turn it into a `chat(message, history)` function

Gradio's chat UI expects a function with exactly this shape: it takes the newest `message` and
the `history` of the conversation so far, and returns the reply as plain text. Inside, we build
the same `messages` list as Step 6 — system prompt, then history, then the new message — and call
the model exactly like before.

In [11]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages)
    return response.choices[0].message.content

In [12]:
chat("I'm ready, let's start the quiz", [])

'Awesome — let’s do it! 🎉\n\nFirst question:  \nWhat’s the quick test for telling a workflow from an agent?'

## NOTE if you are not using OpenAI models

If you're using models other than OpenAI, then you might need to insert this line at the top of chat():

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

## 8. Give it a real chat UI with Gradio

This is the entire chat UI. `gr.ChatInterface` takes any function shaped like
`chat(message, history)` and turns it into a full web-based chat window, handling the history
bookkeeping for you. Run the cell, answer a few questions, get one wrong on purpose, then
**interrupt the cell** (the stop button, or Kernel → Interrupt) when you're done to get your
notebook back.

In [13]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Notice that right now, getting a question wrong doesn't actually do anything besides Study Buddy
*saying* it will remember. Nothing is written down anywhere. That's what the tool in the next
step is for.

## 9. Give your persona a tool

Now for tools. A tool is nothing more than an ordinary Python function. Here's one that records a
missed question to a local file, so the student has a real review sheet to study from later.

In [14]:
def log_missed_question_tool(question):
    print(f"Tool called to log a missed question: {question}")
    with open("missed_questions.txt", "a", encoding="utf-8") as f:
        f.write(question + "\n")
    return "Missed question logged"

In [15]:
log_missed_question_tool("What is an agent loop?")

Tool called to log a missed question: What is an agent loop?


'Missed question logged'

## 10. Describe the tool to the model with JSON

The model can't see your Python code — it only knows a tool exists because you describe it in
JSON: a name, a plain-English description of when to use it, and the shape of the arguments it
takes. This description is what the model reads to decide *when* and *how* to call
`log_missed_question_tool`.

Right now Study Buddy has exactly one tool. Next week we'll see how a persona-based agent can be
given more than one — for now, one tool is plenty to learn the pattern.

In [ ]:
log_missed_question_tool_json = {
    "name": "log_missed_question_tool",
    "description": "Use this tool to record a quiz question the student got wrong or didn't know, so they can review it later",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {"type": "string", "description": "The exact quiz question the student missed"}
        },
        "required": ["question"],
        "additionalProperties": False,
    },
}

tools = [{"type": "function", "function": log_missed_question_tool_json}]

## 11. Update `chat()` to handle one tool call

Pass `tools=tools` into the API call, and the model can now respond by asking for a tool instead
of replying directly — you can tell because `response.choices[0].finish_reason` comes back as
`"tool_calls"` instead of `"stop"`.

The reality is a bit clunky: tool calling is really just an `if` statement. Here we hardcode the
assumption that there's exactly one tool, and it's the missed-question tool. When it fires, we run
`log_missed_question_tool` ourselves, tell the model the result via a `"tool"` message, and call
the model again so it can turn that result into a normal reply, still in character.

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    if response.choices[0].finish_reason == "tool_calls":
        message_obj = response.choices[0].message
        tool_call = message_obj.tool_calls[0]
        question = json.loads(tool_call.function.arguments).get("question")
        log_missed_question_tool(question)
        messages.append(message_obj)
        messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": "Missed question logged"})
        response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    return response.choices[0].message.content

In [3]:
gr.ChatInterface(chat).launch(inbrowser=True)

NameError: name 'gr' is not defined

Try getting a question wrong on purpose. Watch the tool print in your terminal/output, and
check that `missed_questions.txt` gets created next to this notebook with that question inside
it. Interrupt the cell when you're done.

# Important Note!
**Now try this, as the very first message in a fresh chat window** (so there's no quiz history to
confuse things): *"I don't know the answer to either of these. Please log them both as missed:
'What is an agent loop?' and 'What is a harness?'"* That phrasing is deliberately explicit about
wanting both logged, so the model can't reasonably read it any other way.

Don't just look at the chat reply, open `missed_questions.txt` and check what's actually in it.
Step 11's `if` only ever checks for a tool call once, so however the model chooses to ask for two
tool calls (both at once, or one after the other), this `chat()` can't fully keep up with it: you
may get a blank or incomplete-looking reply, an error, or a reply that sounds fine but is quietly
missing something — the one thing you can always count on is that `missed_questions.txt` will be
short a line. That file, not the chat window, is the real evidence.

Keep that message in mind, we'll send it again once Step 12 fixes this.

## 12. The agent loop: handling as many tool calls as needed

Step 11 has a real limitation: it only ever checks for a tool call *once*. Imagine reviewing past
mistakes in one go — "I got the last three questions wrong: agent loop, harness, and tool
schema" — that's three missed questions to log from a single message, and Step 11's code would
only catch the first one. That's exactly what an **agent loop** is for.

Two small changes turn Step 11's `if` into a real loop:
1. `if finish_reason == "tool_calls"` becomes `while finish_reason == "tool_calls"` — keep going
   as long as the model keeps asking for tools.
2. Instead of assuming there's only one tool call, loop over *all* of `message_obj.tool_calls`
   with a `for` loop.

This is your first agent loop, hand-built with no framework — call the model, check if it wants a
tool, run it (possibly several times), call the model again, repeat until it's ready to give a
final answer.

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        message_obj = response.choices[0].message
        messages.append(message_obj)
        for tool_call in message_obj.tool_calls:
            question = json.loads(tool_call.function.arguments).get("question")
            log_missed_question_tool(question)
            messages.append({"role": "tool", "tool_call_id": tool_call.id,
                "content": "Missed question logged"})
        response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    return response.choices[0].message.content

## 13. Try your finished Study Buddy

Same one line as before — but now `chat()` is backed by a real agent loop. Chat with it, try the
"I got the last three wrong" style message, and confirm that every missed question shows up in
`missed_questions.txt` — not just the first one.

In [27]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Tool called to log a missed question: What is an agent loop?
Tool called to log a missed question: What is a harness?


## Wrap-up

Take a moment to look back over what we just built:

- You wrote a **persona** plus a system prompt that turned a generic model into a specific
  character with a job. Nothing about the model itself changed, and it only ever quizzes on the
  notes you gave it.
- `gr.ChatInterface(chat)` gave you a full chat UI for one line of code, as long as
  `chat(message, history)` had the right shape.
- A tool was just a Python function plus a JSON description. The model never runs your code
  directly, your own `if`/`while` logic does.
- The agent loop was two small changes to an `if` statement: check *while* the model wants tools,
  and loop over *all* the tool calls it asks for, not just the first.

There's nothing to submit for this lab. It's practice before the assignment, where you'll build
your own **Persona Agent** for a topic of your choosing: your own persona and notes
instead of Study Buddy's, your own tool instead of `log_missed_question_tool`, but the exact same
persona + system-prompt + Gradio + tool + agent-loop shape we just built here.